In [1]:
import pandas as pd
import numpy as np

# 1. Load historical enriched data
df = pd.read_csv("../data/processed/wmt_pl_enriched.csv")

# Baseline: 2026 Actuals
actuals_2026 = df[df['FiscalYear'] == 2026].iloc[0]
base_rev_2026 = actuals_2026['Revenue']

# 2. Define Driver Assumptions for FY2027
scenarios = {
    'Base Case': {
        'rev_growth': 0.045,      # +4.5% Revenue growth
        'cogs_pct_rev': 0.751,    # COGS holds around 75.1% of Revenue
        'op_inc_growth': 0.040    # Operating Income grows +4.0%
    },
    'Upside (Bull)': {
        'rev_growth': 0.070,      # +7.0% Revenue growth (e.g. strong e-commerce acceleration)
        'cogs_pct_rev': 0.742,    # COGS drops to 74.2% due to supply chain savings
        'op_inc_growth': 0.085    # Operating Income jumps +8.5%
    },
    'Downside (Bear)': {
        'rev_growth': 0.015,      # +1.5% Revenue growth (macro slowdown)
        'cogs_pct_rev': 0.762,    # COGS jumps to 76.2% due to supplier cost inflation
        'op_inc_growth': -0.030   # Operating Income declines -3.0%
    }
}

# 3. Calculate 2027 Projections across Scenarios
results = []

for scenario_name, drivers in scenarios.items():
    proj_rev = base_rev_2026 * (1 + drivers['rev_growth'])
    proj_cogs = proj_rev * drivers['cogs_pct_rev']
    proj_gp = proj_rev - proj_cogs
    proj_op_inc = actuals_2026['OperatingIncome'] * (1 + drivers['op_inc_growth'])
    
    # Calculate key output metrics
    gross_margin = (proj_gp / proj_rev) * 100
    op_margin = (proj_op_inc / proj_rev) * 100
    
    results.append({
        'Scenario': scenario_name,
        'Revenue_$B': round(proj_rev, 2),
        'Rev_Growth_%': round(drivers['rev_growth'] * 100, 2),
        'COGS_$B': round(proj_cogs, 2),
        'GrossProfit_$B': round(proj_gp, 2),
        'GrossMargin_%': round(gross_margin, 2),
        'OpIncome_$B': round(proj_op_inc, 2),
        'OpMargin_%': round(op_margin, 2)
    })

scenario_df = pd.DataFrame(results)

# 4. Save scenario model output to processed data folder
scenario_df.to_csv("../data/processed/wmt_2027_scenarios.csv", index=False)

print("✅ FY2027 Driver-Based Scenario Planning Engine Executed.")
print("\n📊 Projections Summary for FY2027 (in $ Billions):")
print(scenario_df.to_string(index=False))

✅ FY2027 Driver-Based Scenario Planning Engine Executed.

📊 Projections Summary for FY2027 (in $ Billions):
       Scenario  Revenue_$B  Rev_Growth_%  COGS_$B  GrossProfit_$B  GrossMargin_%  OpIncome_$B  OpMargin_%
      Base Case      745.25           4.5   559.68          185.57           24.9        31.01        4.16
  Upside (Bull)      763.08           7.0   566.21          196.87           25.8        32.35        4.24
Downside (Bear)      723.86           1.5   551.58          172.28           23.8        28.93        4.00
